# 1INF52 Deep Learning
## Semestre 2026-2

## Laboratorio 2 - Fundamentos de Redes Neuronales

## Indicaciones generales:

- Duración: **120 minutos**.
- **Las respuestas deben contar con un buen fundamento teórico**, se realizarán descuentos en el puntaje a respuestas que no contesten a lo solicitado.
- Es <font color='red'>OBLIGATORIO DESHABILITAR ASISTENCIA DE IA</font> (Tools->Settings->AI Assistant). De no hacerlo puede ser anulada su evaluacion. Se verificara ello en los  logs y remotamente con AB Tutor.  
- Cualquier indicio de plagio o violacion de las presentes normas resultará en la anulación de la prueba.
- Debe presentar sus respuestas en base a los resultados de ejecución en los casos que se solicite. <br>**No se calificarán aquellas respuestas que no presenten un resultado de ejecución o que no concuerden con este.**
- Subir el cuadernillo con el nombre **Lab2_código.ipynb**, donde código es su código PUCP de 8 dígitos.  
- Se tomará en cuenta en la calificación el uso de **comentarios relevantes**.

## Objetivos:
- Comprender el proceso de entrenamiento de un modelo lineal mediante tensores, diferenciación automática, cálculo de gradientes y descenso de gradiente.
- Implementar y entrenar un modelo lineal desde cero, utilizando minibatches y ajustando parámetros mediante el algoritmo SGD.
- Utilizar las herramientas de PyTorch (TensorDataset, DataLoader, nn.Linear y optimizadores) para implementar el mismo modelo de forma más eficiente y comparar ambas aproximaciones.

# Caso: Regresión Lineal Básica

# Ejercicios Parte 1 (14 ptos)
## Pregunta 1 (12 ptos)

Cada celda `COMPLETAR` con su puntaje indicado entre paréntesis

## Versión sin Pytorch

In [ ]:
import random
import torch
import matplotlib.pyplot as plt

### Conjunto de Datos

Para mantener el problema simple, construiremos un **conjunto de datos artificial** a partir de un modelo lineal al que añadiremos un poco de ruido. Nuestra tarea será **recuperar los parámetros** utilizados para generar los datos.

Generaremos **1000 ejemplos**, cada uno con **2 características** (`features`) obtenidas de una distribución normal estándar. Por lo tanto, tendremos una matriz:

$$
\mathbf{X} \in \mathbb{R}^{1000 \times 2}
$$

Los parámetros verdaderos del modelo serán:

$$
\mathbf{w} =
\begin{bmatrix}
2 \\
-3.4
\end{bmatrix},
\qquad
b = 4.2
$$

Las etiquetas se generarán mediante la siguiente ecuación:

$$
\mathbf{y} = \mathbf{X}\mathbf{w} + b + \boldsymbol{\epsilon}
$$

donde $\boldsymbol{\epsilon}$ representa **ruido gaussiano** con media $0$ y desviación estándar $0.01$. Podemos interpretarlo como un pequeño **error de medición** presente en los datos.

**Objetivo:** a partir de los datos generados, entrenar un modelo que sea capaz de aproximar los parámetros verdaderos $\mathbf{w}$ y $b$.

**Completar la siguiente función. (1 pto)**

In [ ]:
def datos_sinteticos(w, b, n_samples):
    """Genera y = Xw + b + ruido."""
    X = torch.normal(0, 1, (n_samples, len(w)))     # (n, 2)
    y = # COMPLETAR                      # (n,)
    y += torch.normal(0, 0.01, y.shape)             # ruido
    return X, y.reshape(-1, 1)                      # y como columna: (n, 1)

Ahora definimos los parámetros verdaderos y generación del conjunto.

**Completar el siguiente bloque de código. (1 pto)**

**Defina w_real como tensor*

In [ ]:
w_real =  # COMPLETAR
b_real =  # COMPLETAR

X, y =  # COMPLETAR
#Imprimir las dimensiones de X e y
 # COMPLETAR

Cada fila de `X` es un ejemplo de 2 dimensiones y cada fila de `y` es una etiqueta escalar.

**Completar el siguiente bloque de código. (1 pto)**

In [ ]:
# Mostrar un ejemplo de X e y

indice_ej = 0
print(f'Indice {indice_ej}')
print('Features:',    ) # COMPLETAR
print('Label:',      )# COMPLETAR

Si graficamos la segunda característica `X[:, 1]` contra `y`, se ve claramente la relación lineal (con pendiente negativa, porque el peso verdadero es $-3.4$).

**Completar el siguiente bloque de código. (1 pto)**

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(  ,    , s=1) # COMPLETAR
plt.xlabel('Segundo feature')
plt.ylabel('Target')
plt.show()

### Minibatches en Datos

Entrenar un modelo consiste en recorrer el conjunto de datos varias veces. En cada iteración, los datos se dividen en pequeños grupos de ejemplos llamados **minibatches**, que se utilizan para calcular el error y actualizar los parámetros del modelo.

La siguiente función recibe el tamaño del minibatch, las características y las etiquetas. En cada llamada, utiliza `yield` para **entregar un minibatch a la vez**, seleccionando los ejemplos en un orden aleatorio.

**Completar el código en la función. (1 pto)**

In [ ]:
def iterar_minibatch(batch_size, X, y):
    """Devuelve minibatch (X, y) en orden aleatorio."""
    n = len(X)
    indices = list(range(n))

    # Barajamos los ejemplos
    # COMPLETAR

    for i in range(0, n, batch_size):
        batch = torch.tensor(indices[i:i + batch_size])
        yield X[batch], y[batch]

Veamos el primer minibatch. La forma de `X` nos dice el batch size y el número de características; la de `y`, solo el batch size.

In [ ]:
batch_size = 10

for X_batch, y_batch in iterar_minibatch(batch_size, X, y):
    print(X_batch.shape, y_batch.shape)
    print(X_batch)
    print(y_batch)
    break

### Inicializar Parámetros de Modelo

Antes de iniciar el proceso de optimización, necesitamos definir **parámetros iniciales** para el modelo.

Inicializaremos los **pesos** con valores aleatorios obtenidos de una distribución normal con media $0$ y desviación estándar $0.01$. El **sesgo (`bias`)** se inicializará en $0$.

El argumento `requires_grad=True` indica a PyTorch que debe **calcular y almacenar los gradientes** de estos tensores durante la ejecución de las operaciones. Esto permitirá utilizar posteriormente dichos gradientes para actualizar los parámetros mediante el algoritmo de optimización.

**Completar el siguiente bloque de código. (2 ptos)**

**Asegúrese que w tenga forma (len(w_real), 1)*

In [ ]:
w =  # COMPLETAR
b =  # COMPLETAR

w, b

### Definir el Modelo

La salida de un modelo lineal se obtiene mediante el **producto matriz-vector** entre las características $\mathbf{X}$ y los pesos $\mathbf{w}$, al que se suma el sesgo $b$:

$$
\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b
$$

En PyTorch, `X @ w` calcula el producto matriz-vector y produce un vector con las predicciones. El término `b` es un escalar que se suma a cada componente del vector mediante **broadcasting**, sin necesidad de replicarlo manualmente.

**Completar la siguiente función. (1 pto)**

In [ ]:
def modelo_lineal(X, w, b):
    """Modelo de regresión lineal."""
    return  # COMPLETAR

### Definir Loss

Para medir el error entre las predicciones y los valores reales, utilizaremos el **error cuadrático**:

$$
l(\hat{y}, y) = \frac{1}{2}(\hat{y} - y)^2
$$

El método `reshape` nos permite asegurarnos de que `y` tenga la **misma forma** que `y_hat` antes de realizar la operación.

Esto es importante porque, si una variable tuviera forma `(n,)` y la otra `(n, 1)`, PyTorch aplicaría *broadcasting* automáticamente y el resultado tendría forma `(n, n)`, en lugar de calcular el error correspondiente entre cada predicción y su etiqueta.

**Completar la siguiente función. (1.5 ptos)**

In [ ]:
def squared_loss(y_hat, y):
    """Pérdida cuadrática (una por ejemplo, sin promediar)."""
    return # COMPLETAR

### Definir Algoritmo de Optimización

En cada paso del entrenamiento, calculamos el **gradiente de la pérdida** utilizando un minibatch y actualizamos los parámetros en la dirección que permite reducirla:

$$
\theta \leftarrow \theta - \frac{\eta}{|\mathcal{B}|}\nabla_{\theta} l
$$

En esta actualización debemos tener en cuenta tres aspectos importantes:

- `torch.no_grad()`: la actualización de los parámetros **no debe formar parte del cálculo del gradiente**. Por ello, utilizamos este contexto para indicarle a PyTorch que no registre estas operaciones en el grafo computacional.

- **División entre `batch_size`**: la función de pérdida utilizada suma los errores de todos los ejemplos del minibatch. Dividir el gradiente entre `batch_size` permite obtener un promedio, de modo que el tamaño de la actualización no dependa directamente del número de ejemplos del lote.

- `p.grad.zero_()`: PyTorch **acumula los gradientes** por defecto. Por ello, después de actualizar los parámetros, debemos poner sus gradientes en cero antes de procesar el siguiente minibatch.



In [ ]:
def sgd(params, lr, batch_size):
    """Descenso de gradiente estocástico por minibatch."""
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad / batch_size
            p.grad.zero_()

### Entrenamiento

Ya tenemos todas las piezas necesarias para construir el **bucle de entrenamiento**. Este esquema aparece, con pequeñas variaciones, de forma recurrente en problemas de *deep learning*:

1. **Inicializar los parámetros** $\mathbf{w}$ y $b$.

2. **Repetir durante varias épocas**:
   - Para cada minibatch:
     - Calcular las predicciones y la pérdida (**paso hacia adelante** o *forward pass*).
     - Calcular los gradientes mediante `l.sum().backward()` (**paso hacia atrás** o *backward pass*).
     - Actualizar los parámetros utilizando `sgd`.

Una **época** corresponde a un recorrido completo por todo el conjunto de datos. El número de épocas `num_epocas` y la tasa de aprendizaje `lr` son **hiperparámetros**, es decir, valores que debemos definir antes o durante el proceso de entrenamiento.

**Completar el siguiente bloque de código. (0.5 ptos)**

In [ ]:
# Definir los hiperparámetros. Considerar la tasa de aprendizaje = 0.03 y  3 épocas



**Completar el siguiente bloque de código. (2 ptos)**

In [ ]:
historial = []

for epoca in range(num_epocas):
    for X_batch, y_batch in iterar_minibatch(batch_size, X, y):
        l =  # COMPLETAR loss(modelo, y_batch)
        l.sum().backward()              # Calcula dl/dw y dl/db
        sgd([w, b], lr, batch_size)     # Actualiza w y b

    # Al terminar la época, medimos la pérdida sobre todos los datos
    with torch.no_grad():
        total_loss = squared_loss(modelo_lineal(X, w, b), y).mean()
    historial.append(float(total_loss))
    print(f'época {epoca + 1}, pérdida {float(total_loss):f}')

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(range(1, num_epocas + 1), historial, marker='o')
plt.xlabel('época')
plt.ylabel('pérdida')
plt.show()

### Comparar Parámetros (Real vs Ajustado)

Como generamos los datos nosotros mismos, sabemos exactamente cuáles eran los
parámetros verdaderos y podemos medir qué tan cerca quedamos.

In [ ]:
print(f'error en w: {w_real - w.reshape(w_real.shape)}')
print(f'error en b: {b_real - b}')

No debemos asumir que siempre será posible recuperar los parámetros exactos con los que se generaron los datos.

En *machine learning*, nuestro objetivo generalmente no es recuperar los parámetros **“verdaderos”**, sino encontrar parámetros que permitan al modelo **realizar buenas predicciones sobre datos nuevos**.

En otras palabras, nos interesa más el **desempeño predictivo del modelo** que la recuperación exacta de los parámetros utilizados para generar los datos.

### Resumen

- Es posible entrenar un modelo **desde cero** utilizando únicamente tensores y diferenciación automática, sin depender de capas u optimizadores predefinidos.

- El ciclo de entrenamiento sigue una estructura fundamental que se repite en *deep learning*: **predecir → calcular la pérdida → `backward()` → actualizar los parámetros → poner los gradientes en cero**.

## Pregunta 2 (2 ptos)

Pruebe cambiando el **número de características** del conjunto de datos. Por ejemplo, utilice **5 características** y, por lo tanto, 5 pesos verdaderos.

Compruebe que el código desarrollado anteriormente **sigue funcionando correctamente sin necesidad de realizar cambios en el algoritmo**. Comente brevemente sus observaciones.

In [ ]:
# Coloque aquí su código de prueba
#Defina w_real5 y b_real5
#Definir y5
#Inicializar w5 y b5
#Training
#Imprimir error

**Observaciones:**

# Ejercicios Parte 2 (6 ptos)
## Pregunta 3 (4 ptos)
Completar el código faltante donde se indique con la palabra `COMPLETAR`.

# Versión con Pytorch

Todo lo anterior lo implementamos **manualmente** con el objetivo de comprender qué ocurre durante el entrenamiento de un modelo. Sin embargo, en la práctica no es necesario programar cada componente desde cero: los frameworks de *deep learning* proporcionan **capas, funciones de pérdida y optimizadores** que simplifican considerablemente este proceso.

A continuación, repetiremos **exactamente el mismo experimento**, pero utilizando `torch.nn` y las herramientas de PyTorch. Compara ambas implementaciones: cada componente que construimos manualmente tiene un equivalente dentro del framework.

| Implementación manual | Con PyTorch |
|---|---|
| `iterar_minibatch` | `DataLoader` |
| `w`, `b` con `requires_grad=True` | `nn.Linear(2, 1)` — crea y registra los parámetros automáticamente |
| `modelo_lineal(X, w, b)` | `net(X)` |
| `squared_loss` | `nn.MSELoss()` |
| `sgd([w, b], lr, batch_size)` | `optimizador.step()` |
| `p.grad.zero_()` | `optimizador.zero_grad()` |

Esta comparación permite observar cómo **PyTorch abstrae los detalles que implementamos manualmente**, manteniendo la misma lógica fundamental del entrenamiento.

In [ ]:
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

### Leer los Datos

`TensorDataset` permite **empaquetar las características y sus etiquetas** como un único conjunto de datos. Por su parte, `DataLoader` se encarga de **organizar los datos en minibatches** y, si se indica `shuffle=True`, de **barajarlos aleatoriamente** en cada época.

De esta manera, `DataLoader` reemplaza nuestra función `iterar_minibatch` y simplifica la gestión de los datos durante el entrenamiento.

**Completar el siguiente bloque de código. (1.5 ptos)**

In [ ]:
dataset = TensorDataset(X, y)
iterador =  # COMPLETAR

X_batch, y_batch = next(iter(iterador))
X_batch.shape, y_batch.shape

### Definir el Modelo

`nn.Linear(2, 1)` representa una **capa totalmente conectada** con 2 entradas y 1 salida. Internamente, crea y registra automáticamente una matriz de pesos `weight` con forma `(1, 2)` y un `bias` con forma `(1,)`. Ambos parámetros tienen `requires_grad=True`, por lo que PyTorch calculará sus gradientes durante el entrenamiento.

`nn.Sequential` permite **encadenar varias capas y operaciones en secuencia**, de modo que la salida de una se convierte en la entrada de la siguiente. En este caso solo utilizaremos una capa, pero mantendremos esta estructura porque es una forma habitual y sencilla de construir redes neuronales más complejas.

**Completar el siguiente bloque de código. (1.5 ptos)**

In [ ]:
red_nn =  # COMPLETAR

# Inicializamos igual que antes: pesos ~ N(0, 0.01) y sesgo en 0
red_nn[0].weight.data.normal_(0, 0.01)
red_nn[0].bias.data.fill_(0)

red_nn

### Loss y Optimizador

- `nn.MSELoss()` calcula el **error cuadrático medio** (*Mean Squared Error*). A diferencia de nuestra función `squared_loss`, que calculaba el error de cada ejemplo como $\frac{1}{2}(\hat{y}-y)^2$, `nn.MSELoss()` **promedia los errores del minibatch y no incluye el factor $\frac{1}{2}$**.

  Por esta razón, al utilizar `nn.MSELoss()` no necesitamos dividir manualmente el gradiente entre el tamaño del batch durante la actualización: el gradiente que recibe el optimizador ya corresponde a la pérdida promedio del minibatch.

- `torch.optim.SGD` implementa el algoritmo de **Descenso de Gradiente Estocástico (SGD)**. Recibe los parámetros que debe optimizar —en este caso, `red_nn.parameters()`, que recopila automáticamente los parámetros registrados por la red— y la tasa de aprendizaje `lr`.

**Completar el siguiente bloque de código. (1 pto)**

In [ ]:
perdida_nn = # COMPLETAR
optimizador = torch.optim.SGD(red_nn.parameters(), lr=0.03)

### Entrenamiento

El bucle de entrenamiento mantiene la **misma lógica fundamental** que en la implementación manual, pero ahora algunas tareas son gestionadas automáticamente por el optimizador:

1. `optimizador.zero_grad()` pone en cero los gradientes acumulados de los parámetros. En la implementación manual, esta operación la realizábamos dentro de `sgd`.

2. `l.backward()` calcula los gradientes mediante diferenciación automática. Utilizamos `backward()` directamente porque `MSELoss()` devuelve una **pérdida escalar** al promediar los errores del minibatch. En la implementación manual, utilizábamos `l.sum().backward()` porque nuestra función de pérdida devolvía un valor por cada ejemplo.

3. `optimizador.step()` utiliza los gradientes calculados para **actualizar automáticamente los parámetros** del modelo de acuerdo con el algoritmo de optimización seleccionado.

In [ ]:
num_epocas = 3

for epoca in range(num_epocas):
    for X_batch, y_batch in iterador:
        l = perdida_nn(red_nn(X_batch), y_batch)
        optimizador.zero_grad()     # 1. limpiar gradientes
        l.backward()                # 2. calcular gradientes
        optimizador.step()          # 3. actualizar parámetros

    with torch.no_grad():
        l_total = perdida_nn(red_nn(X), y)
    print(f'época {epoca + 1}, pérdida {float(l_total):f}')

Y comparamos otra vez con los parámetros verdaderos. Los pesos aprendidos están en `red_nn[0].weight` y el sesgo en `red_nn[0].bias`.

In [ ]:
w_aprendido = red_nn[0].weight.data
b_aprendido = red_nn[0].bias.data

print(f'error en w: {w_real - w_aprendido.reshape(w_real.shape)}')
print(f'error en b: {b_real - b_aprendido}')

El resultado es el mismo, pero con **mucho menos código**. La ventaja de haber implementado primero el entrenamiento de forma manual es que ahora podemos entender qué ocurre internamente: `nn.Linear` y `optimizador.step()` dejan de ser **“cajas negras”**, porque conocemos las operaciones que realizan y cómo participan en el proceso de entrenamiento.

## Pregunta 4 (2 ptos)

Vuelva a definir el modelo usando `nn.Linear(2, 1, bias=False)`, es decir, una capa **sin término de sesgo**. Entrene nuevamente con el mismo código y la misma tasa de aprendizaje.

Compare el **error en `w`** con los del modelo original. ¿Logra el modelo ajustar bien los datos?

Calcule además $b_{real}^2$ y compárelo con la *pérdida final*.


In [ ]:
# Coloque aquí su código de prueba
#Defina red_sin_bias
#Defina la función de perdida
#Defina el optimizador con SGD y lr
# Defina num de epocas
#Entrene el modelo
#Compare con el modelo de la pregunta 3


**Observaciones:**